In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import time
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score,f1_score,precision_score,recall_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D,MaxPooling2D,Flatten, Dense,Dropout,BatchNormalization
from tensorflow.keras.optimizers import Adam ,SGD ,RMSprop,Nadam
from tensorflow.keras.preprocessing.image import  ImageDataGenerator,load_img, img_to_array

2025-08-05 17:32:12.287577: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754415132.647836      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754415132.749474      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


# What is `ImageDataGenerator`?

In deep learning for images, we usually have thousands of images.  
Loading all images into memory at once is expensive.

`ImageDataGenerator` in TensorFlow / Keras is a tool that:

- Loads images batch by batch during training
- Preprocesses images automatically
- Applies augmentation to create modified versions of images
- Feeds images to the CNN model continuously

So instead of manually reading and editing images, the generator does it automatically while training.

---

# First Line Explanation

```python
train_gen = ImageDataGenerator(
    rescale=1/255,
    rotation_range=20,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    fill_mode="nearest"
)
```

This creates a **training image generator**.

Its job is:

1. Read training images
2. Apply preprocessing
3. Apply augmentation
4. Send batches to the model

---

# Detailed Explanation of Each Parameter

## 1. `rescale=1/255`

Images are originally pixel values between:

```python
0 → 255
```

Example pixel:

```python
[120, 200, 255]
```

After scaling:

```python
[120/255, 200/255, 255/255]
=
[0.47, 0.78, 1.0]
```

So all pixel values become between:

```python
0 → 1
```

Why?

Because neural networks train better with small normalized values.

---

## 2. `rotation_range=20`

Randomly rotates images during training.

The image can rotate between:

```python
-20 degrees to +20 degrees
```

Example:

Original brain MRI:
- straight image

Augmented versions:
- slightly tilted left
- slightly tilted right

Why?

To make the model robust to rotated images.

---

## 3. `zoom_range=0.15`

Random zoom in/out.

```python
0.15 = 15%
```

The image may become:

- slightly zoomed in
- slightly zoomed out

Why?

Helps the model recognize tumors at different scales.

---

## 4. `width_shift_range=0.1`

Moves image horizontally.

```python
0.1 = 10% of image width
```

If image width is:

```python
224 pixels
```

Then shifting may be:

```python
22 pixels right or left
```

---

## 5. `height_shift_range=0.1`

Moves image vertically.

Again:

```python
10% of image height
```

So the image may move:

- upward
- downward

Why shifts are useful?

Real images are not always perfectly centered.

---

## 6. `horizontal_flip=True`

Randomly flips images horizontally.

Example:

```text
Original → Mirrored
```

Why?

Creates more training examples.

---

## 7. `fill_mode="nearest"`

After rotation or shifting, empty spaces appear.

Example after rotation:

```text
██████
█    █
█ IMG█
█    █
██████
```

Empty corners appear.

`fill_mode="nearest"` fills missing pixels using nearest neighboring pixels.

This prevents black empty regions.

---

# Important Idea About Augmentation

The generator does NOT permanently create new files.

Instead:

During every training batch:

1. Original image loaded
2. Random transformations applied
3. Modified image sent to model

So every epoch the model may see different versions of the same image.

---

# Second Generator

```python
valid_gen = ImageDataGenerator(rescale=1/255)
```

Validation images are ONLY normalized.

No augmentation.

Why?

Because validation should evaluate real unchanged images.

---

# Now This Important Line

```python
train_aug = train_gen.flow_from_dataframe(
    train_df,
    x_col="image_path",
    y_col="labels",
    target_size=(224,224),
    color_mode="rgb",
    class_mode="categorical",
    batch_size=16,
    shuffle=True
)
```

This line connects the dataframe with the generator.

It tells the generator:

```text
"Read image paths from dataframe,
load images,
apply augmentation,
create batches,
and return them to the model."
```

---

# Detailed Explanation

## `train_df`

This dataframe contains:

| image_path | labels |
|---|---|
| img1.jpg | glioma |
| img2.jpg | meningioma |

So:

- first column → image location
- second column → class label

---

## `x_col="image_path"`

Means:

```python
Use column "image_path" as image file paths
```

Example:

```python
"C:/dataset/img1.jpg"
```

---

## `y_col="labels"`

Means:

```python
Use column "labels" as target classes
```

Example:

```python
glioma
pituitary
notumor
```

---

## `target_size=(224,224)`

Resize every image into:

```python
224 × 224
```

Why?

CNN models require fixed-size input.

So even if original images differ:

```python
300×400
512×512
100×200
```

all become:

```python
224×224
```

---

## `color_mode="rgb"`

Load image as RGB.

So image shape becomes:

```python
(224,224,3)
```

3 channels:

- Red
- Green
- Blue

---

## `class_mode="categorical"`

Used for multiclass classification.

You have:

```python
4 classes
```

So labels become one-hot encoded.

Example:

| Class | Encoding |
|---|---|
| glioma | [1,0,0,0] |
| meningioma | [0,1,0,0] |
| pituitary | [0,0,1,0] |
| no_tumor | [0,0,0,1] |

This works with:

```python
Dense(4, activation='softmax')
```

and:

```python
categorical_crossentropy
```

---

## `batch_size=16`

The generator sends images in groups of 16.

Instead of:

```python
5712 images at once
```

it sends:

```python
16 images each step
```

So one batch shape becomes:

```python
(16,224,224,3)
```

---

## `shuffle=True`

Randomly shuffles training data every epoch.

Why?

Prevents model from memorizing image order.

Very important during training.

---

# What Does This Output Mean?

```python
Found 5712 validated image filenames belonging to 4 classes.
```

Meaning:

- The generator successfully found:

```python
5712 training images
```

- All image paths are valid
- Images belong to:

```python
4 classes
```

Similarly:

```python
Found 655 validated image filenames belonging to 4 classes.
```

Means:

- Validation generator found:

```python
655 validation images
```

- Across same 4 categories

---

# Final Pipeline Summary

```text
DataFrame
   ↓
flow_from_dataframe()
   ↓
Load image paths
   ↓
Resize images
   ↓
Normalize pixels
   ↓
Apply augmentation
   ↓
Create batches
   ↓
Send batches to CNN
```

The generator output dimension is mainly:

(batch_size, image_height, image_width, channels)

In your case:

batch_size = 16
target_size = (224,224)
color_mode = "rgb"

So one batch shape becomes:

(16, 224, 224, 3)

Meaning:

16 → number of images in one batch
224 → image height
224 → image width
3 → RGB channels

In [2]:
data="/kaggle/input/brain-tumor-mri-dataset/Training"
image_paths=[]
labels=[]
folders=os.listdir(data)
for folder in folders:
    folder_path=os.path.join(data,folder)
    images=os.listdir(folder_path)
    for image in images:
        image_path=os.path.join(folder_path,image)
        image_paths.append(image_path)
        labels.append(folder)



        

In [3]:
train_df=pd.DataFrame({"image_path":image_paths,"labels":labels})
train_df


,image_path,labels
0,/kaggle/input/brain-tumor-mri-dataset/Training...,pituitary
1,/kaggle/input/brain-tumor-mri-dataset/Training...,pituitary
2,/kaggle/input/brain-tumor-mri-dataset/Training...,pituitary
3,/kaggle/input/brain-tumor-mri-dataset/Training...,pituitary
4,/kaggle/input/brain-tumor-mri-dataset/Training...,pituitary
...,...,...
5707,/kaggle/input/brain-tumor-mri-dataset/Training...,glioma
5708,/kaggle/input/brain-tumor-mri-dataset/Training...,glioma
5709,/kaggle/input/brain-tumor-mri-dataset/Training...,glioma
5710,/kaggle/input/brain-tumor-mri-dataset/Training...,glioma


In [4]:
test_data="/kaggle/input/brain-tumor-mri-dataset/Testing"
image_paths_test=[]
labels_test=[]
folders=os.listdir(test_data)
for folder in folders:
    folder_path=os.path.join(test_data,folder)
    images=os.listdir(folder_path)
    for image in images:
        image_path=os.path.join(folder_path,image)
        image_paths_test.append(image_path)
        labels_test.append(folder)
        



In [5]:
test_df=pd.DataFrame({"image path":image_paths_test,"labels":labels_test})
test_df

,image path,labels
0,/kaggle/input/brain-tumor-mri-dataset/Testing/...,pituitary
1,/kaggle/input/brain-tumor-mri-dataset/Testing/...,pituitary
2,/kaggle/input/brain-tumor-mri-dataset/Testing/...,pituitary
3,/kaggle/input/brain-tumor-mri-dataset/Testing/...,pituitary
4,/kaggle/input/brain-tumor-mri-dataset/Testing/...,pituitary
...,...,...
1306,/kaggle/input/brain-tumor-mri-dataset/Testing/...,glioma
1307,/kaggle/input/brain-tumor-mri-dataset/Testing/...,glioma
1308,/kaggle/input/brain-tumor-mri-dataset/Testing/...,glioma
1309,/kaggle/input/brain-tumor-mri-dataset/Testing/...,glioma


In [6]:
valid_df,test_df=train_test_split(test_df,random_state=42,test_size=0.5)




In [7]:
data_gen=ImageDataGenerator(rescale=1/255)
train_noaug=data_gen.flow_from_dataframe(train_df,x_col='image_path', y_col='labels',target_size=(224,224),color_mode="rgb",class_mode="categorical",batch_size=16)
valid_noaug=data_gen.flow_from_dataframe(valid_df,x_col='image path', y_col='labels',target_size=(224,224),color_mode="rgb",class_mode="categorical",batch_size=8)
test_noaug=data_gen.flow_from_dataframe(test_df,x_col='image path', y_col='labels',target_size=(224,224),color_mode="rgb",class_mode="categorical",batch_size=8)

Found 5712 validated image filenames belonging to 4 classes.
Found 655 validated image filenames belonging to 4 classes.
Found 656 validated image filenames belonging to 4 classes.


In [8]:
train_gen=ImageDataGenerator(rescale=1/255,rotation_range=20,zoom_range=0.15,width_shift_range=0.1,height_shift_range=0.1,horizontal_flip=True,fill_mode="nearest")
valid_gen=ImageDataGenerator(rescale=1/255)
test_gen=ImageDataGenerator(rescale=1/255)
train_aug=train_gen.flow_from_dataframe(train_df,x_col="image_path",y_col="labels",target_size=(224,224),color_mode="rgb",class_mode="categorical",batch_size=16,shuffle=True)
valid_aug=valid_gen.flow_from_dataframe(valid_df,x_col="image path",y_col="labels",target_size=(224,224),color_mode="rgb",class_mode="categorical",batch_size=8)



Found 5712 validated image filenames belonging to 4 classes.
Found 655 validated image filenames belonging to 4 classes.


In [9]:
test_aug=test_gen.flow_from_dataframe(test_df,x_col="image path",y_col="labels",target_size=(224,224),color_mode="rgb",class_mode="categorical",batch_size=8,shuffle=False)

Found 656 validated image filenames belonging to 4 classes.


In [10]:
import random
import tensorflow as tf

# Set global seed
seed = 42
np.random.seed(seed)
tf.random.set_seed(seed)
random.seed(seed)


In [54]:
model1_notaug=Sequential([
    Conv2D(32,(3,3),input_shape=(224,224,3),activation="relu"),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(32,activation="relu"),
    Dense(4,activation="softmax"),

])
model1_notaug.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 394272)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │    12,616,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,617,764 (48.13 MB)

 Trainable params: 12,617,764 (48.13 MB)

 Non-trainable params: 0 (0.00 B)

In [55]:
start_time_noaug=time.time()
model1_notaug.compile(optimizer=SGD(momentum=0.9,learning_rate=0.1),loss="categorical_crossentropy",metrics=['accuracy'])
model1_notaug.fit(train_noaug,validation_data=valid_noaug,epochs=10,batch_size=16)
average_train_time_per_epoch_noaug=(time.time()-start_time_noaug)/10


Epoch 1/10


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


357/357 ━━━━━━━━━━━━━━━━━━━━ 16s 42ms/step - accuracy: 0.2635 - loss: 1.4366 - val_accuracy: 0.3008 - val_loss: 1.3854
Epoch 2/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 15s 41ms/step - accuracy: 0.2653 - loss: 1.3968 - val_accuracy: 0.3008 - val_loss: 1.3817
Epoch 3/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 14s 40ms/step - accuracy: 0.2861 - loss: 1.3875 - val_accuracy: 0.2275 - val_loss: 1.3931
Epoch 4/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 15s 41ms/step - accuracy: 0.2781 - loss: 1.3946 - val_accuracy: 0.2305 - val_loss: 1.4285
Epoch 5/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 14s 40ms/step - accuracy: 0.2475 - loss: 1.4015 - val_accuracy: 0.2305 - val_loss: 1.3991
Epoch 6/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 15s 42ms/step - accuracy: 0.2452 - loss: 1.4008 - val_accuracy: 0.2275 - val_loss: 1.3911
Epoch 7/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 16s 44ms/step - accuracy: 0.2512 - loss: 1.3989 - val_accuracy: 0.3008 - val_loss: 1.3916
Epoch 8/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 15s 41ms/step - accuracy: 0.2556 - loss: 1.3952 - val_accurac

In [56]:
test_loss_notaug1,test_accu_notaug1 = model1_notaug.evaluate(test_noaug)
print(f"Accuracy: {test_accu_notaug1 * 100:.2f}%")
print("average training time is ",average_train_time_per_epoch_noaug)

82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.2472 - loss: 1.3891
Accuracy: 22.56%
average training time is  14.89887249469757


In [57]:
model1=Sequential([
    Conv2D(32,(3,3),input_shape=(224,224,3),activation="relu"),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(32,activation="relu"),
    Dense(4,activation="softmax"),

])
model1.summary()


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 394272)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 32)             │    12,616,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,617,764 (48.13 MB)

 Trainable params: 12,617,764 (48.13 MB)

 Non-trainable params: 0 (0.00 B)

In [58]:
start_time=time.time()
model1.compile(optimizer=SGD(momentum=0.9,learning_rate=0.1),loss="categorical_crossentropy",metrics=['accuracy'])
model1.fit(train_aug,validation_data=valid_aug,epochs=10,batch_size=16)
average_train_time_per_epoch=(time.time()-start_time)/10



Epoch 1/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 66s 181ms/step - accuracy: 0.2519 - loss: 2.7549 - val_accuracy: 0.3008 - val_loss: 1.3827
Epoch 2/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 63s 177ms/step - accuracy: 0.2581 - loss: 1.3968 - val_accuracy: 0.2412 - val_loss: 1.3878
Epoch 3/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 63s 177ms/step - accuracy: 0.2719 - loss: 1.3909 - val_accuracy: 0.2412 - val_loss: 1.3896
Epoch 4/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 63s 177ms/step - accuracy: 0.2515 - loss: 1.3933 - val_accuracy: 0.2305 - val_loss: 1.3857
Epoch 5/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 63s 178ms/step - accuracy: 0.2724 - loss: 1.3924 - val_accuracy: 0.3008 - val_loss: 1.3813
Epoch 6/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 63s 177ms/step - accuracy: 0.2590 - loss: 1.3961 - val_accuracy: 0.2412 - val_loss: 1.3858
Epoch 7/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 64s 178ms/step - accuracy: 0.2572 - loss: 1.3937 - val_accuracy: 0.2305 - val_loss: 1.3984
Epoch 8/10
357/357 ━━━━━━━━━━━━━━━━━━━━ 63s 177ms/step - accuracy: 0.2546 - loss: 1

In [59]:
test_loss1,test_accu1 = model1.evaluate(test_aug)
valid_loss1,valid_accu1=model1.evaluate(valid_aug)
training_loss1,training_accu1=model1.evaluate(train_aug)
print(f"validation Accuracy: {valid_accu1* 100:.2f}%")
print(f"validation Loss: {valid_loss1}")
print(f"training Accuracy: {training_accu1* 100:.2f}%")
print(f"training Loss: {training_loss1}")
print(f"Accuracy: {test_accu1 * 100:.2f}%")
print("average training time is ",average_train_time_per_epoch)

82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.2402 - loss: 1.3927
82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.2617 - loss: 1.3911
357/357 ━━━━━━━━━━━━━━━━━━━━ 62s 172ms/step - accuracy: 0.2637 - loss: 1.3884
validation Accuracy: 22.75%
validation Loss: 1.3970425128936768
training Accuracy: 25.51%
training Loss: 1.390657663345337
Accuracy: 23.02%
average training time is  63.675496792793275


In [60]:
model2=Sequential([
    Conv2D(32,(3,3),input_shape=(224,224,3),activation="relu"),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(32,activation="relu"),
    Dense(4,activation="softmax"),

])
model2.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_5 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_5 (Flatten)             │ (None, 394272)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │    12,616,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,617,764 (48.13 MB)

 Trainable params: 12,617,764 (48.13 MB)

 Non-trainable params: 0 (0.00 B)

In [61]:
start_time2=time.time()
model2.compile(optimizer=SGD(momentum=0.9,learning_rate=0.1),loss="categorical_crossentropy",metrics=['accuracy'])
model2.fit(train_aug,validation_data=valid_aug,epochs=15,batch_size=16)
average_train_time_per_epoch2=(time.time()-start_time2)/15


Epoch 1/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 180ms/step - accuracy: 0.2625 - loss: 2.0805 - val_accuracy: 0.3008 - val_loss: 1.3828
Epoch 2/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 64s 179ms/step - accuracy: 0.2557 - loss: 1.3970 - val_accuracy: 0.3008 - val_loss: 1.3825
Epoch 3/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 64s 178ms/step - accuracy: 0.2620 - loss: 1.3911 - val_accuracy: 0.2305 - val_loss: 1.3910
Epoch 4/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 63s 177ms/step - accuracy: 0.2399 - loss: 1.3963 - val_accuracy: 0.3008 - val_loss: 1.3838
Epoch 5/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 64s 178ms/step - accuracy: 0.2618 - loss: 1.3939 - val_accuracy: 0.2275 - val_loss: 1.4015
Epoch 6/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 64s 179ms/step - accuracy: 0.2488 - loss: 1.3946 - val_accuracy: 0.2275 - val_loss: 1.3963
Epoch 7/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 64s 179ms/step - accuracy: 0.2591 - loss: 1.3965 - val_accuracy: 0.2275 - val_loss: 1.3909
Epoch 8/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 183ms/step - accuracy: 0.2622 - loss: 1

In [62]:
test_loss2,test_accu2 = model2.evaluate(test_aug)
valid_loss2,valid_accu2=model2.evaluate(valid_aug)
training_loss2,training_accu2=model2.evaluate(train_aug)
print(f"validation Accuracy: {valid_accu2* 100:.2f}%")
print(f"validation Loss: {valid_loss2}")
print(f"training Accuracy: {training_accu2* 100:.2f}%")
print(f"training Loss: {training_loss2}")
print(f"Accuracy: {test_accu2 * 100:.2f}%")
print("average training time is ",average_train_time_per_epoch2)



82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - accuracy: 0.3165 - loss: 1.3844
82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.2672 - loss: 1.4000
357/357 ━━━━━━━━━━━━━━━━━━━━ 62s 172ms/step - accuracy: 0.2841 - loss: 1.3891
validation Accuracy: 30.08%
validation Loss: 1.3870781660079956
training Accuracy: 27.92%
training Loss: 1.3898251056671143
Accuracy: 31.71%
average training time is  64.26958847045898


In [65]:
model3=Sequential([
    Conv2D(32,(3,3),input_shape=(224,224,3),activation="relu"),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(32,activation="relu"),
    Dense(4,activation="softmax"),

])
model3.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_7 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_7 (Flatten)             │ (None, 394272)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 32)             │    12,616,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,617,764 (48.13 MB)

 Trainable params: 12,617,764 (48.13 MB)

 Non-trainable params: 0 (0.00 B)

In [66]:
start_time3=time.time()
model3.compile(optimizer=SGD(momentum=0.9,learning_rate=0.01),loss="categorical_crossentropy",metrics=['accuracy'])
model3.fit(train_aug,validation_data=valid_aug,epochs=15,batch_size=16)
average_train_time_per_epoch3=(time.time()-start_time3)/15

Epoch 1/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 71s 195ms/step - accuracy: 0.4482 - loss: 1.2058 - val_accuracy: 0.5817 - val_loss: 1.0604
Epoch 2/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 64s 179ms/step - accuracy: 0.5965 - loss: 0.9990 - val_accuracy: 0.6595 - val_loss: 0.8854
Epoch 3/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 64s 180ms/step - accuracy: 0.5930 - loss: 0.9516 - val_accuracy: 0.5252 - val_loss: 1.2103
Epoch 4/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 181ms/step - accuracy: 0.5608 - loss: 1.0013 - val_accuracy: 0.5313 - val_loss: 1.2814
Epoch 5/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 64s 179ms/step - accuracy: 0.5711 - loss: 0.9536 - val_accuracy: 0.6137 - val_loss: 0.9914
Epoch 6/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 181ms/step - accuracy: 0.5849 - loss: 0.9544 - val_accuracy: 0.6321 - val_loss: 0.9593
Epoch 7/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 182ms/step - accuracy: 0.5987 - loss: 0.9215 - val_accuracy: 0.6153 - val_loss: 1.1202
Epoch 8/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 64s 179ms/step - accuracy: 0.6096 - loss: 0

In [67]:
test_loss3,test_accu3 = model3.evaluate(test_aug)
valid_loss3,valid_accu3=model3.evaluate(valid_aug)
training_loss3,training_accu3=model3.evaluate(train_aug)
print(f"validation Accuracy: {valid_accu3* 100:.2f}%")
print(f"validation Loss: {valid_loss3}")
print(f"training Accuracy: {training_accu3* 100:.2f}%")
print(f"training Loss: {training_loss3}")
print(f"Accuracy: {test_accu3 * 100:.2f}%")
print("average training time is ",average_train_time_per_epoch3)

82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - accuracy: 0.6309 - loss: 0.8848
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.6386 - loss: 0.8264
357/357 ━━━━━━━━━━━━━━━━━━━━ 62s 173ms/step - accuracy: 0.6439 - loss: 0.8214
validation Accuracy: 65.04%
validation Loss: 0.8240594267845154
training Accuracy: 64.53%
training Loss: 0.8226485252380371
Accuracy: 64.94%
average training time is  64.77311285336812


In [68]:
model=Sequential([
    Conv2D(32,(3,3),input_shape=(224,224,3),activation="relu"),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(32,activation="relu"),
    Dense(4,activation="softmax"),

])
model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_8 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_8 (Flatten)             │ (None, 394272)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 32)             │    12,616,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,617,764 (48.13 MB)

 Trainable params: 12,617,764 (48.13 MB)

 Non-trainable params: 0 (0.00 B)

In [69]:
start_time_=time.time()
model.compile(optimizer=SGD(momentum=0.9,learning_rate=0.001),loss="categorical_crossentropy",metrics=['accuracy'])
model.fit(train_aug,validation_data=valid_aug,epochs=15,batch_size=16)
average_train_time_per_epoch_=(time.time()-start_time_)/15

Epoch 1/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 68s 188ms/step - accuracy: 0.3897 - loss: 1.2486 - val_accuracy: 0.6397 - val_loss: 0.9563
Epoch 2/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 64s 178ms/step - accuracy: 0.6540 - loss: 0.9093 - val_accuracy: 0.6763 - val_loss: 0.8281
Epoch 3/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 64s 179ms/step - accuracy: 0.6917 - loss: 0.7968 - val_accuracy: 0.6748 - val_loss: 0.8260
Epoch 4/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 66s 184ms/step - accuracy: 0.7122 - loss: 0.7282 - val_accuracy: 0.6824 - val_loss: 0.8875
Epoch 5/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 183ms/step - accuracy: 0.7165 - loss: 0.7183 - val_accuracy: 0.7435 - val_loss: 0.6249
Epoch 6/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 64s 179ms/step - accuracy: 0.7326 - loss: 0.6676 - val_accuracy: 0.7542 - val_loss: 0.6524
Epoch 7/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 64s 179ms/step - accuracy: 0.7509 - loss: 0.6456 - val_accuracy: 0.7145 - val_loss: 0.8034
Epoch 8/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 64s 179ms/step - accuracy: 0.7410 - loss: 0

In [70]:
test_loss_,test_accu_ = model.evaluate(test_aug)
valid_loss_,valid_accu_=model.evaluate(valid_aug)
training_loss_,training_accu_=model.evaluate(train_aug)
print(f"validation Accuracy: {valid_accu_* 100:.2f}%")
print(f"validation Loss: {valid_loss_}")
print(f"training Accuracy: {training_accu_* 100:.2f}%")
print(f"training Loss: {training_loss_}")
print(f"Accuracy: {test_accu_ * 100:.2f}%")
print("average training time is ",average_train_time_per_epoch_)

82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.6870 - loss: 0.7715
82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.7122 - loss: 0.7531
357/357 ━━━━━━━━━━━━━━━━━━━━ 63s 175ms/step - accuracy: 0.7627 - loss: 0.5946
validation Accuracy: 72.06%
validation Loss: 0.7656731009483337
training Accuracy: 76.87%
training Loss: 0.574054479598999
Accuracy: 70.12%
average training time is  64.67944920857748


In [71]:
model4=Sequential([
    Conv2D(32,(3,3),input_shape=(224,224,3),activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(64,(3,3),activation="relu"),
    
    Flatten(),
    Dense(64,activation="relu"),
    Dense(4,activation="softmax"),

])
model4.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_9 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_9 (Flatten)             │ (None, 760384)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 64)             │    48,664,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 48,684,292 (185.72 MB)

 Trainable params: 48,684,292 (185.72 MB)

 Non-trainable params: 0 (0.00 B)

In [72]:
start_time4=time.time()
model4.compile(optimizer=SGD(momentum=0.9,learning_rate=0.001),loss="categorical_crossentropy",metrics=['accuracy'])
model4.fit(train_aug,validation_data=valid_aug,epochs=15,batch_size=16)
average_train_time_per_epoch4=(time.time()-start_time4)/15

Epoch 1/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 73s 192ms/step - accuracy: 0.4467 - loss: 1.1759 - val_accuracy: 0.5924 - val_loss: 0.9300
Epoch 2/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 181ms/step - accuracy: 0.6378 - loss: 0.8954 - val_accuracy: 0.7008 - val_loss: 0.8619
Epoch 3/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 66s 184ms/step - accuracy: 0.6912 - loss: 0.7830 - val_accuracy: 0.7237 - val_loss: 0.7702
Epoch 4/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 182ms/step - accuracy: 0.7255 - loss: 0.7121 - val_accuracy: 0.7344 - val_loss: 0.6747
Epoch 5/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 182ms/step - accuracy: 0.7305 - loss: 0.6848 - val_accuracy: 0.7649 - val_loss: 0.6555
Epoch 6/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 181ms/step - accuracy: 0.7466 - loss: 0.6530 - val_accuracy: 0.7557 - val_loss: 0.6040
Epoch 7/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 181ms/step - accuracy: 0.7531 - loss: 0.6398 - val_accuracy: 0.7069 - val_loss: 0.7724
Epoch 8/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 182ms/step - accuracy: 0.7788 - loss: 0

In [73]:
test_loss4,test_accu4 = model4.evaluate(test_aug)
valid_loss4,valid_accu4=model4.evaluate(valid_aug)
training_loss4,training_accu4=model4.evaluate(train_aug)
print(f"validation Accuracy: {valid_accu4* 100:.2f}%")
print(f"validation Loss: {valid_loss4}")
print(f"training Accuracy: {training_accu4* 100:.2f}%")
print(f"training Loss: {training_loss4}")
print(f"Accuracy: {test_accu4 * 100:.2f}%")
print("average training time is ",average_train_time_per_epoch4)

82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.7777 - loss: 0.6157
82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.7752 - loss: 0.5714
357/357 ━━━━━━━━━━━━━━━━━━━━ 62s 173ms/step - accuracy: 0.8289 - loss: 0.4393
validation Accuracy: 80.61%
validation Loss: 0.5113533735275269
training Accuracy: 82.04%
training Loss: 0.4568692445755005
Accuracy: 80.64%
average training time is  65.61859463055929


In [74]:
model5=Sequential([
    Conv2D(32,(3,3),input_shape=(224,224,3),activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(64,(3,3),activation="relu"),
    
    Flatten(),
    Dense(32,activation="relu"),
    Dense(64,activation="relu"),
    Dense(4,activation="softmax"),

])
model5.summary()

Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_11 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_10 (Flatten)            │ (None, 760384)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 32)             │    24,332,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,354,084 (92.90 MB)

 Trainable params: 24,354,084 (92.90 MB)

 Non-trainable params: 0 (0.00 B)

In [75]:
start_time5=time.time()
model5.compile(optimizer=SGD(momentum=0.9,learning_rate=0.001),loss="categorical_crossentropy",metrics=['accuracy'])
model5.fit(train_aug,validation_data=valid_aug,epochs=15,batch_size=16)
average_train_time_per_epoch5=(time.time()-start_time5)/15

Epoch 1/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 70s 188ms/step - accuracy: 0.3538 - loss: 1.2649 - val_accuracy: 0.5893 - val_loss: 1.0054
Epoch 2/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 182ms/step - accuracy: 0.6485 - loss: 0.8735 - val_accuracy: 0.6901 - val_loss: 0.7563
Epoch 3/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 183ms/step - accuracy: 0.6728 - loss: 0.7870 - val_accuracy: 0.7023 - val_loss: 0.7888
Epoch 4/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 182ms/step - accuracy: 0.6985 - loss: 0.7548 - val_accuracy: 0.7298 - val_loss: 0.6668
Epoch 5/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 182ms/step - accuracy: 0.7305 - loss: 0.6876 - val_accuracy: 0.7160 - val_loss: 0.7035
Epoch 6/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 67s 187ms/step - accuracy: 0.7458 - loss: 0.6399 - val_accuracy: 0.7527 - val_loss: 0.5767
Epoch 7/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 182ms/step - accuracy: 0.7573 - loss: 0.6215 - val_accuracy: 0.7405 - val_loss: 0.5851
Epoch 8/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 181ms/step - accuracy: 0.7796 - loss: 0

In [76]:
test_loss5,test_accu5 = model5.evaluate(test_aug)
valid_loss5,valid_accu5=model5.evaluate(valid_aug)
training_loss5,training_accu5=model5.evaluate(train_aug)
print(f"validation Accuracy: {valid_accu5* 100:.2f}%")
print(f"validation Loss: {valid_loss5}")
print(f"training Accuracy: {training_accu5* 100:.2f}%")
print(f"training Loss: {training_loss5}")
print(f"Accuracy: {test_accu5 * 100:.2f}%")
print("average training time is ",average_train_time_per_epoch5)

82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.7957 - loss: 0.5121
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.8030 - loss: 0.5020
357/357 ━━━━━━━━━━━━━━━━━━━━ 63s 175ms/step - accuracy: 0.8144 - loss: 0.4762
validation Accuracy: 80.31%
validation Loss: 0.5037648677825928
training Accuracy: 81.39%
training Loss: 0.47870001196861267
Accuracy: 79.12%
average training time is  65.56300746599833


In [77]:
model6=Sequential([
    Conv2D(32,(3,3),input_shape=(224,224,3),activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(64,(3,3),activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(128,(3,3),activation="relu"),
    
    
    Flatten(),
    Dense(64,activation="relu"),
    Dense(4,activation="softmax"),

])
model6.summary()

Model: "sequential_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_13 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_11 (Flatten)            │ (None, 346112)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 64)             │    22,151,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,244,740 (84.86 MB)

 Trainable params: 22,244,740 (84.86 MB)

 Non-trainable params: 0 (0.00 B)

In [78]:
start_time6=time.time()
model6.compile(optimizer=SGD(momentum=0.9,learning_rate=0.001),loss="categorical_crossentropy",metrics=['accuracy'])
model6.fit(train_aug,validation_data=valid_aug,epochs=15,batch_size=16)
average_train_time_per_epoch6=(time.time()-start_time6)/15

Epoch 1/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 70s 187ms/step - accuracy: 0.4041 - loss: 1.2023 - val_accuracy: 0.5389 - val_loss: 1.0478
Epoch 2/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 183ms/step - accuracy: 0.6307 - loss: 0.9035 - val_accuracy: 0.6855 - val_loss: 0.7923
Epoch 3/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 66s 183ms/step - accuracy: 0.6642 - loss: 0.8387 - val_accuracy: 0.6580 - val_loss: 1.0480
Epoch 4/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 181ms/step - accuracy: 0.7024 - loss: 0.7598 - val_accuracy: 0.6931 - val_loss: 0.7348
Epoch 5/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 181ms/step - accuracy: 0.7297 - loss: 0.6778 - val_accuracy: 0.6870 - val_loss: 0.9251
Epoch 6/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 183ms/step - accuracy: 0.7426 - loss: 0.6495 - val_accuracy: 0.6855 - val_loss: 0.8479
Epoch 7/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 64s 180ms/step - accuracy: 0.7610 - loss: 0.6074 - val_accuracy: 0.7359 - val_loss: 0.6745
Epoch 8/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 64s 180ms/step - accuracy: 0.7622 - loss: 0

In [79]:
test_loss6,test_accu6 = model6.evaluate(test_aug)
valid_loss6,valid_accu6=model6.evaluate(valid_aug)
training_loss6,training_accu6=model6.evaluate(train_aug)
print(f"validation Accuracy: {valid_accu6* 100:.2f}%")
print(f"validation Loss: {valid_loss6}")
print(f"training Accuracy: {training_accu6* 100:.2f}%")
print(f"training Loss: {training_loss6}")
print(f"Accuracy: {test_accu6 * 100:.2f}%")
print("average training time is ",average_train_time_per_epoch6)

82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.8118 - loss: 0.4223
82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.8166 - loss: 0.4125
357/357 ━━━━━━━━━━━━━━━━━━━━ 63s 174ms/step - accuracy: 0.8349 - loss: 0.4106
validation Accuracy: 81.68%
validation Loss: 0.44319090247154236
training Accuracy: 82.86%
training Loss: 0.42934510111808777
Accuracy: 81.25%
average training time is  65.34270044962565


In [80]:
model7=Sequential([
    Conv2D(32,(3,3),input_shape=(224,224,3),activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(64,(3,3),activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(128,(3,3),activation="relu"),
    
    
    Flatten(),
    Dense(64,activation="relu"),
    Dense(4,activation="softmax"),

])
model7.summary()

Model: "sequential_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_16 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_17 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_18 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_12 (Flatten)            │ (None, 346112)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 64)             │    22,151,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,244,740 (84.86 MB)

 Trainable params: 22,244,740 (84.86 MB)

 Non-trainable params: 0 (0.00 B)

In [81]:
start_time7=time.time()
model7.compile(optimizer=SGD(momentum=0.9,learning_rate=0.001),loss="categorical_crossentropy",metrics=['accuracy'])
model7.fit(train_aug,validation_data=valid_aug,epochs=15,batch_size=32)
average_train_time_per_epoch7=(time.time()-start_time7)/15

Epoch 1/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 68s 186ms/step - accuracy: 0.4231 - loss: 1.1970 - val_accuracy: 0.6672 - val_loss: 0.8953
Epoch 2/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 66s 185ms/step - accuracy: 0.6581 - loss: 0.8584 - val_accuracy: 0.6962 - val_loss: 0.8060
Epoch 3/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 183ms/step - accuracy: 0.6851 - loss: 0.7852 - val_accuracy: 0.7206 - val_loss: 0.7259
Epoch 4/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 66s 184ms/step - accuracy: 0.7125 - loss: 0.7144 - val_accuracy: 0.7389 - val_loss: 0.7005
Epoch 5/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 66s 183ms/step - accuracy: 0.7361 - loss: 0.6747 - val_accuracy: 0.7573 - val_loss: 0.5878
Epoch 6/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 183ms/step - accuracy: 0.7631 - loss: 0.5989 - val_accuracy: 0.7084 - val_loss: 0.6761
Epoch 7/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 67s 187ms/step - accuracy: 0.7633 - loss: 0.5914 - val_accuracy: 0.7588 - val_loss: 0.6183
Epoch 8/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 66s 186ms/step - accuracy: 0.7670 - loss: 0

In [82]:
test_loss7,test_accu7 = model7.evaluate(test_aug)
valid_loss7,valid_accu7=model7.evaluate(valid_aug)
training_loss7,training_accu7=model7.evaluate(train_aug)
print(f"validation Accuracy: {valid_accu7* 100:.2f}%")
print(f"validation Loss: {valid_loss7}")
print(f"training Accuracy: {training_accu7* 100:.2f}%")
print(f"training Loss: {training_loss7}")
print(f"Accuracy: {test_accu7 * 100:.2f}%")
print("average training time is ",average_train_time_per_epoch7)

82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.7950 - loss: 0.5616
82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.7990 - loss: 0.5190
357/357 ━━━━━━━━━━━━━━━━━━━━ 64s 178ms/step - accuracy: 0.8314 - loss: 0.4228
validation Accuracy: 79.85%
validation Loss: 0.5457869172096252
training Accuracy: 82.98%
training Loss: 0.4288859963417053
Accuracy: 78.35%
average training time is  66.1844860871633


In [83]:
model8=Sequential([
    Conv2D(32,(3,3),input_shape=(224,224,3),activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(64,(3,3),activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(128,(3,3),activation="relu"),
    
    
    Flatten(),
    Dense(64,activation="relu"),
    Dense(4,activation="softmax"),

])
model8.summary()

Model: "sequential_13"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_19 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_15 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_20 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_16 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_21 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_13 (Flatten)            │ (None, 346112)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 64)             │    22,151,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,244,740 (84.86 MB)

 Trainable params: 22,244,740 (84.86 MB)

 Non-trainable params: 0 (0.00 B)

In [84]:
start_time8=time.time()
model8.compile(optimizer=SGD(momentum=0.9,learning_rate=0.001),loss="categorical_crossentropy",metrics=['accuracy'])
model8.fit(train_aug,validation_data=valid_aug,epochs=15,batch_size=64)
average_train_time_per_epoch8=(time.time()-start_time8)/15

Epoch 1/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 69s 189ms/step - accuracy: 0.4260 - loss: 1.1940 - val_accuracy: 0.6412 - val_loss: 0.8885
Epoch 2/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 67s 187ms/step - accuracy: 0.6479 - loss: 0.8827 - val_accuracy: 0.6580 - val_loss: 0.8497
Epoch 3/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 68s 190ms/step - accuracy: 0.6908 - loss: 0.7904 - val_accuracy: 0.6580 - val_loss: 0.9006
Epoch 4/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 67s 187ms/step - accuracy: 0.7122 - loss: 0.7367 - val_accuracy: 0.7328 - val_loss: 0.6711
Epoch 5/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 66s 185ms/step - accuracy: 0.7455 - loss: 0.6452 - val_accuracy: 0.7679 - val_loss: 0.5834
Epoch 6/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 66s 184ms/step - accuracy: 0.7502 - loss: 0.6292 - val_accuracy: 0.7221 - val_loss: 0.6775
Epoch 7/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 66s 186ms/step - accuracy: 0.7573 - loss: 0.6067 - val_accuracy: 0.7786 - val_loss: 0.6201
Epoch 8/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 67s 187ms/step - accuracy: 0.7753 - loss: 0

In [85]:
test_loss8,test_accu8 = model8.evaluate(test_aug)
valid_loss8,valid_accu8=model8.evaluate(valid_aug)
training_loss8,training_accu8=model8.evaluate(train_aug)
print(f"validation Accuracy: {valid_accu8* 100:.2f}%")
print(f"validation Loss: {valid_loss8}")
print(f"training Accuracy: {training_accu8* 100:.2f}%")
print(f"training Loss: {training_loss8}")
print(f"Accuracy: {test_accu8 * 100:.2f}%")
print("average training time is ",average_train_time_per_epoch8)

82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.7983 - loss: 0.5303
82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.7691 - loss: 0.6337
357/357 ━━━━━━━━━━━━━━━━━━━━ 64s 177ms/step - accuracy: 0.8321 - loss: 0.4373
validation Accuracy: 77.40%
validation Loss: 0.5819054245948792
training Accuracy: 83.07%
training Loss: 0.43329790234565735
Accuracy: 80.18%
average training time is  66.47521212895711


In [87]:
model9=Sequential([
    Conv2D(32,(3,3),input_shape=(224,224,3),activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(64,(3,3),activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(128,(3,3),activation="relu"),
    
    
    Flatten(),
    Dense(64,activation="sigmoid"),
    Dense(4,activation="softmax"),

])
model9.summary()

Model: "sequential_14"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_22 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_17 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_23 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_18 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_24 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_14 (Flatten)            │ (None, 346112)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 64)             │    22,151,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_30 (Dense)                │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,244,740 (84.86 MB)

 Trainable params: 22,244,740 (84.86 MB)

 Non-trainable params: 0 (0.00 B)

In [88]:
start_time9=time.time()
model9.compile(optimizer=SGD(momentum=0.9,learning_rate=0.001),loss="categorical_crossentropy",metrics=['accuracy'])
model9.fit(train_aug,validation_data=valid_aug,epochs=15,batch_size=16)
average_train_time_per_epoch9=(time.time()-start_time9)/15

Epoch 1/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 69s 189ms/step - accuracy: 0.4264 - loss: 1.2393 - val_accuracy: 0.6489 - val_loss: 0.9903
Epoch 2/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 66s 186ms/step - accuracy: 0.6162 - loss: 0.9526 - val_accuracy: 0.6641 - val_loss: 0.8678
Epoch 3/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 183ms/step - accuracy: 0.6683 - loss: 0.8519 - val_accuracy: 0.6779 - val_loss: 0.8452
Epoch 4/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 183ms/step - accuracy: 0.6698 - loss: 0.8309 - val_accuracy: 0.6763 - val_loss: 0.8184
Epoch 5/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 66s 184ms/step - accuracy: 0.6850 - loss: 0.7938 - val_accuracy: 0.7038 - val_loss: 0.7583
Epoch 6/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 66s 184ms/step - accuracy: 0.7133 - loss: 0.7514 - val_accuracy: 0.7115 - val_loss: 0.7509
Epoch 7/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 66s 184ms/step - accuracy: 0.7152 - loss: 0.7326 - val_accuracy: 0.6931 - val_loss: 0.7519
Epoch 8/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 183ms/step - accuracy: 0.7135 - loss: 0

In [89]:
test_loss9,test_accu9 = model9.evaluate(test_aug)
valid_loss9,valid_accu9=model9.evaluate(valid_aug)
training_loss9,training_accu9=model9.evaluate(train_aug)
print(f"validation Accuracy: {valid_accu9* 100:.2f}%")
print(f"validation Loss: {valid_loss9}")
print(f"training Accuracy: {training_accu9* 100:.2f}%")
print(f"training Loss: {training_loss9}")
print(f"Accuracy: {test_accu9 * 100:.2f}%")
print("average training time is ",average_train_time_per_epoch9)

82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.7536 - loss: 0.6431
82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.7432 - loss: 0.6353
357/357 ━━━━━━━━━━━━━━━━━━━━ 66s 185ms/step - accuracy: 0.7716 - loss: 0.5811
validation Accuracy: 72.98%
validation Loss: 0.6626780033111572
training Accuracy: 77.78%
training Loss: 0.5755214095115662
Accuracy: 73.78%
average training time is  65.74656947453816


In [90]:
model10=Sequential([
    Conv2D(32,(3,3),input_shape=(224,224,3),activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(64,(3,3),activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(128,(3,3),activation="relu"),
    
    
    Flatten(),
    Dense(64,activation="swish"),
    Dense(4,activation="softmax"),

])
model10.summary()

Model: "sequential_15"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_25 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_19 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_26 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_20 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_27 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_15 (Flatten)            │ (None, 346112)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_31 (Dense)                │ (None, 64)             │    22,151,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_32 (Dense)                │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,244,740 (84.86 MB)

 Trainable params: 22,244,740 (84.86 MB)

 Non-trainable params: 0 (0.00 B)

In [91]:
start_time10=time.time()
model10.compile(optimizer=SGD(momentum=0.9,learning_rate=0.001),loss="categorical_crossentropy",metrics=['accuracy'])
model10.fit(train_aug,validation_data=valid_aug,epochs=15,batch_size=16)
average_train_time_per_epoch10=(time.time()-start_time10)/15

Epoch 1/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 68s 186ms/step - accuracy: 0.4156 - loss: 1.1928 - val_accuracy: 0.6794 - val_loss: 0.8324
Epoch 2/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 66s 183ms/step - accuracy: 0.6799 - loss: 0.8104 - val_accuracy: 0.6489 - val_loss: 0.7801
Epoch 3/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 67s 187ms/step - accuracy: 0.7070 - loss: 0.7446 - val_accuracy: 0.7069 - val_loss: 0.7107
Epoch 4/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 67s 188ms/step - accuracy: 0.7279 - loss: 0.6636 - val_accuracy: 0.6885 - val_loss: 0.7861
Epoch 5/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 183ms/step - accuracy: 0.7412 - loss: 0.6360 - val_accuracy: 0.7603 - val_loss: 0.5909
Epoch 6/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 183ms/step - accuracy: 0.7719 - loss: 0.5912 - val_accuracy: 0.7756 - val_loss: 0.6336
Epoch 7/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 67s 188ms/step - accuracy: 0.7850 - loss: 0.5734 - val_accuracy: 0.7542 - val_loss: 0.7326
Epoch 8/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 67s 186ms/step - accuracy: 0.7857 - loss: 0

In [92]:
test_loss10,test_accu10 = model10.evaluate(test_aug)
valid_loss10,valid_accu10=model10.evaluate(valid_aug)
training_loss10,training_accu10=model10.evaluate(train_aug)
print(f"validation Accuracy: {valid_accu10* 100:.2f}%")
print(f"validation Loss: {valid_loss10}")
print(f"training Accuracy: {training_accu10* 100:.2f}%")
print(f"training Loss: {training_loss10}")
print(f"Accuracy: {test_accu10 * 100:.2f}%")
print("average training time is ",average_train_time_per_epoch10)

82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.7976 - loss: 0.5082
82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.8052 - loss: 0.5794
357/357 ━━━━━━━━━━━━━━━━━━━━ 62s 174ms/step - accuracy: 0.8443 - loss: 0.3961
validation Accuracy: 80.46%
validation Loss: 0.5739591121673584
training Accuracy: 84.05%
training Loss: 0.40567225217819214
Accuracy: 79.88%
average training time is  66.14813696543375


In [130]:
model11=Sequential([
    Conv2D(32,(3,3),input_shape=(224,224,3),activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(64,(3,3),activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(128,(3,3),activation="relu"),
    
    
    Flatten(),
    Dense(64,activation="relu"),
    Dense(4,activation="softmax"),

])
model11.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_27"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_61 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_43 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_62 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_44 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_63 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_27 (Flatten)            │ (None, 346112)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_55 (Dense)                │ (None, 64)             │    22,151,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_56 (Dense)                │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,244,740 (84.86 MB)

 Trainable params: 22,244,740 (84.86 MB)

 Non-trainable params: 0 (0.00 B)

In [94]:
start_time11=time.time()
model11.compile(optimizer=RMSprop(learning_rate=0.001),loss="categorical_crossentropy",metrics=['accuracy'])
model11.fit(train_aug,validation_data=valid_aug,epochs=15,batch_size=16)
average_train_time_per_epoch11=(time.time()-start_time11)/15

Epoch 1/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 67s 183ms/step - accuracy: 0.4921 - loss: 1.6009 - val_accuracy: 0.7069 - val_loss: 0.6891
Epoch 2/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 66s 185ms/step - accuracy: 0.7217 - loss: 0.7106 - val_accuracy: 0.6275 - val_loss: 1.2739
Epoch 3/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 67s 186ms/step - accuracy: 0.7579 - loss: 0.6104 - val_accuracy: 0.7221 - val_loss: 0.7606
Epoch 4/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 182ms/step - accuracy: 0.7924 - loss: 0.5304 - val_accuracy: 0.7389 - val_loss: 0.6995
Epoch 5/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 181ms/step - accuracy: 0.7997 - loss: 0.5084 - val_accuracy: 0.7115 - val_loss: 0.6255
Epoch 6/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 183ms/step - accuracy: 0.8125 - loss: 0.4786 - val_accuracy: 0.7374 - val_loss: 0.8578
Epoch 7/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 181ms/step - accuracy: 0.8350 - loss: 0.4340 - val_accuracy: 0.7893 - val_loss: 0.4941
Epoch 8/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 182ms/step - accuracy: 0.8419 - loss: 0

In [95]:
test_loss11,test_accu11 = model11.evaluate(test_aug)
valid_loss11,valid_accu11=model11.evaluate(valid_aug)
training_loss11,training_accu11=model11.evaluate(train_aug)
print(f"validation Accuracy: {valid_accu11* 100:.2f}%")
print(f"validation Loss: {valid_loss11}")
print(f"training Accuracy: {training_accu11* 100:.2f}%")
print(f"training Loss: {training_loss11}")
print(f"Accuracy: {test_accu11* 100:.2f}%")
print("average training time is ",average_train_time_per_epoch11)

82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.8535 - loss: 0.5255
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.8867 - loss: 0.3092
357/357 ━━━━━━━━━━━━━━━━━━━━ 63s 175ms/step - accuracy: 0.8911 - loss: 0.2974
validation Accuracy: 87.48%
validation Loss: 0.3536965250968933
training Accuracy: 89.02%
training Loss: 0.29440000653266907
Accuracy: 86.74%
average training time is  65.02487607002259


In [96]:
modeladam=Sequential([
    Conv2D(32,(3,3),input_shape=(224,224,3),activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(64,(3,3),activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(128,(3,3),activation="relu"),
    
    
    Flatten(),
    Dense(64,activation="relu"),
    Dense(4,activation="softmax"),

])
modeladam.summary()

Model: "sequential_17"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_31 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_23 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_32 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_24 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_33 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_17 (Flatten)            │ (None, 346112)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_35 (Dense)                │ (None, 64)             │    22,151,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_36 (Dense)                │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,244,740 (84.86 MB)

 Trainable params: 22,244,740 (84.86 MB)

 Non-trainable params: 0 (0.00 B)

In [97]:
start_timeadam=time.time()
modeladam.compile(optimizer=Adam(learning_rate=0.001),loss="categorical_crossentropy",metrics=['accuracy'])
modeladam.fit(train_aug,validation_data=valid_aug,epochs=15,batch_size=16)
average_train_time_per_epochadam=(time.time()-start_time11)/15

Epoch 1/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 69s 184ms/step - accuracy: 0.5039 - loss: 1.2986 - val_accuracy: 0.4733 - val_loss: 1.3851
Epoch 2/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 182ms/step - accuracy: 0.7251 - loss: 0.6712 - val_accuracy: 0.6702 - val_loss: 0.8548
Epoch 3/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 183ms/step - accuracy: 0.7633 - loss: 0.6138 - val_accuracy: 0.7618 - val_loss: 0.5605
Epoch 4/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 183ms/step - accuracy: 0.7824 - loss: 0.5578 - val_accuracy: 0.7389 - val_loss: 0.6716
Epoch 5/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 182ms/step - accuracy: 0.7951 - loss: 0.5104 - val_accuracy: 0.7435 - val_loss: 0.6572
Epoch 6/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 182ms/step - accuracy: 0.7986 - loss: 0.5011 - val_accuracy: 0.8000 - val_loss: 0.4687
Epoch 7/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 181ms/step - accuracy: 0.8211 - loss: 0.4644 - val_accuracy: 0.7313 - val_loss: 0.7863
Epoch 8/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 181ms/step - accuracy: 0.8236 - loss: 0

In [98]:
test_lossadam,test_accuadam = modeladam.evaluate(test_aug)
valid_lossadam,valid_accuadam=modeladam.evaluate(valid_aug)
training_lossadam,training_accuadam=modeladam.evaluate(train_aug)
print(f"validation Accuracy: {valid_accuadam* 100:.2f}%")
print(f"validation Loss: {valid_lossadam}")
print(f"training Accuracy: {training_accuadam* 100:.2f}%")
print(f"training Loss: {training_lossadam}")
print(f"Accuracy: {test_accuadam* 100:.2f}%")
print("average training time is ",average_train_time_per_epochadam)

82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.8186 - loss: 0.4568
82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.8213 - loss: 0.4215
357/357 ━━━━━━━━━━━━━━━━━━━━ 63s 175ms/step - accuracy: 0.8845 - loss: 0.3091
validation Accuracy: 81.98%
validation Loss: 0.4315302073955536
training Accuracy: 87.55%
training Loss: 0.32328882813453674
Accuracy: 82.01%
average training time is  143.0333310763041


In [101]:
modelnadam=Sequential([
    Conv2D(32,(3,3),input_shape=(224,224,3),activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(64,(3,3),activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(128,(3,3),activation="relu"),
    
    
    Flatten(),
    Dense(64,activation="relu"),
    Dense(4,activation="softmax"),

])
modelnadam.summary()

Model: "sequential_19"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_37 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_27 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_38 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_28 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_39 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_19 (Flatten)            │ (None, 346112)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_39 (Dense)                │ (None, 64)             │    22,151,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_40 (Dense)                │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,244,740 (84.86 MB)

 Trainable params: 22,244,740 (84.86 MB)

 Non-trainable params: 0 (0.00 B)

In [103]:
start_timenadam=time.time()
modelnadam.compile(optimizer=Nadam(learning_rate=0.001),loss="categorical_crossentropy",metrics=['accuracy'])
modelnadam.fit(train_aug,validation_data=valid_aug,epochs=15,batch_size=16)
average_train_time_per_epochnadam=(time.time()-start_timenadam)/15

Epoch 1/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 71s 189ms/step - accuracy: 0.5364 - loss: 1.1703 - val_accuracy: 0.6824 - val_loss: 0.9697
Epoch 2/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 67s 188ms/step - accuracy: 0.7491 - loss: 0.6354 - val_accuracy: 0.7328 - val_loss: 0.8023
Epoch 3/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 67s 188ms/step - accuracy: 0.7868 - loss: 0.5640 - val_accuracy: 0.7191 - val_loss: 0.8625
Epoch 4/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 68s 191ms/step - accuracy: 0.7993 - loss: 0.5185 - val_accuracy: 0.7023 - val_loss: 0.7773
Epoch 5/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 67s 188ms/step - accuracy: 0.8053 - loss: 0.4740 - val_accuracy: 0.7832 - val_loss: 0.5149
Epoch 6/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 67s 189ms/step - accuracy: 0.8309 - loss: 0.4305 - val_accuracy: 0.7573 - val_loss: 0.6038
Epoch 7/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 69s 193ms/step - accuracy: 0.8446 - loss: 0.4226 - val_accuracy: 0.7740 - val_loss: 0.5697
Epoch 8/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 70s 195ms/step - accuracy: 0.8422 - loss: 0

In [104]:
test_lossnadam,test_accunadam = modelnadam.evaluate(test_aug)
valid_lossnadam,valid_accunadam=modelnadam.evaluate(valid_aug)
training_lossnadam,training_accunadam=modelnadam.evaluate(train_aug)
print(f"validation Accuracy: {valid_accunadam* 100:.2f}%")
print(f"validation Loss: {valid_lossnadam}")
print(f"training Accuracy: {training_accunadam* 100:.2f}%")
print(f"training Loss: {training_lossnadam}")
print(f"Accuracy: {test_accunadam* 100:.2f}%")
print("average training time is ",average_train_time_per_epochnadam)

82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.8480 - loss: 0.3905
82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.8667 - loss: 0.3231
357/357 ━━━━━━━━━━━━━━━━━━━━ 62s 173ms/step - accuracy: 0.9019 - loss: 0.2601
validation Accuracy: 85.04%
validation Loss: 0.3729918897151947
training Accuracy: 90.41%
training Loss: 0.2538716793060303
Accuracy: 84.76%
average training time is  67.36949283281962


In [105]:
model12=Sequential([
    Conv2D(32,(3,3),input_shape=(224,224,3),activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(64,(3,3),activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(128,(3,3),activation="relu"),
    
    
    Flatten(),
    Dense(64,activation="relu"),
    Dropout(0.15),
    Dense(4,activation="softmax"),

])
model12.summary()

Model: "sequential_20"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_40 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_29 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_41 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_30 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_42 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_20 (Flatten)            │ (None, 346112)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_41 (Dense)                │ (None, 64)             │    22,151,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_42 (Dense)                │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,244,740 (84.86 MB)

 Trainable params: 22,244,740 (84.86 MB)

 Non-trainable params: 0 (0.00 B)

In [106]:
start_time12=time.time()
model12.compile(optimizer=RMSprop(learning_rate=0.001),loss="categorical_crossentropy",metrics=['accuracy'])
model12.fit(train_aug,validation_data=valid_aug,epochs=15,batch_size=16)
average_train_time_per_epoch12=(time.time()-start_time12)/15

Epoch 1/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 70s 190ms/step - accuracy: 0.4422 - loss: 1.5081 - val_accuracy: 0.7176 - val_loss: 0.7033
Epoch 2/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 66s 183ms/step - accuracy: 0.7006 - loss: 0.7475 - val_accuracy: 0.7084 - val_loss: 0.7936
Epoch 3/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 182ms/step - accuracy: 0.7384 - loss: 0.6457 - val_accuracy: 0.7084 - val_loss: 0.7404
Epoch 4/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 66s 184ms/step - accuracy: 0.7843 - loss: 0.5607 - val_accuracy: 0.7756 - val_loss: 0.5849
Epoch 5/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 66s 184ms/step - accuracy: 0.7973 - loss: 0.5374 - val_accuracy: 0.7847 - val_loss: 0.5160
Epoch 6/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 183ms/step - accuracy: 0.8016 - loss: 0.5140 - val_accuracy: 0.7863 - val_loss: 0.5431
Epoch 7/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 181ms/step - accuracy: 0.8090 - loss: 0.4809 - val_accuracy: 0.7664 - val_loss: 0.5779
Epoch 8/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 183ms/step - accuracy: 0.8220 - loss: 0

In [107]:
test_loss12,test_accu12 = model12.evaluate(test_aug)
valid_loss12,valid_accu12=model12.evaluate(valid_aug)
training_loss12,training_accu12=model12.evaluate(train_aug)
print(f"validation Accuracy: {valid_accu12* 100:.2f}%")
print(f"validation Loss: {valid_loss12}")
print(f"training Accuracy: {training_accu12* 100:.2f}%")
print(f"training Loss: {training_loss12}")
print(f"Accuracy: {test_accu12* 100:.2f}%")
print("average training time is ",average_train_time_per_epoch12)

82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.8344 - loss: 0.4558
82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.8459 - loss: 0.4737
357/357 ━━━━━━━━━━━━━━━━━━━━ 62s 172ms/step - accuracy: 0.8669 - loss: 0.3794
validation Accuracy: 84.43%
validation Loss: 0.488845556974411
training Accuracy: 86.96%
training Loss: 0.3716295659542084
Accuracy: 83.54%
average training time is  65.36769444147745


In [111]:
model13=Sequential([
    Conv2D(32,(3,3),input_shape=(224,224,3),activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(64,(3,3),activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(128,(3,3),activation="relu"),
    
    
    Flatten(),
    Dense(64,activation="relu"),
    Dropout(0.5),
    Dense(4,activation="softmax"),

])
model13.summary()

Model: "sequential_22"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_46 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_33 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_47 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_34 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_48 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_22 (Flatten)            │ (None, 346112)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_45 (Dense)                │ (None, 64)             │    22,151,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_46 (Dense)                │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,244,740 (84.86 MB)

 Trainable params: 22,244,740 (84.86 MB)

 Non-trainable params: 0 (0.00 B)

In [112]:
start_time13=time.time()
model13.compile(optimizer=RMSprop(learning_rate=0.001),loss="categorical_crossentropy",metrics=['accuracy'])
model13.fit(train_aug,validation_data=valid_aug,epochs=15,batch_size=16)
average_train_time_per_epoch13=(time.time()-start_time13)/15

Epoch 1/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 68s 184ms/step - accuracy: 0.4400 - loss: 1.2839 - val_accuracy: 0.6840 - val_loss: 0.8532
Epoch 2/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 183ms/step - accuracy: 0.6450 - loss: 0.8822 - val_accuracy: 0.7557 - val_loss: 0.6895
Epoch 3/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 181ms/step - accuracy: 0.6853 - loss: 0.7910 - val_accuracy: 0.7527 - val_loss: 0.6597
Epoch 4/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 181ms/step - accuracy: 0.7065 - loss: 0.7566 - val_accuracy: 0.7664 - val_loss: 0.7352
Epoch 5/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 182ms/step - accuracy: 0.7282 - loss: 0.6861 - val_accuracy: 0.7893 - val_loss: 0.5243
Epoch 6/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 64s 180ms/step - accuracy: 0.7453 - loss: 0.6537 - val_accuracy: 0.7634 - val_loss: 0.6214
Epoch 7/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 182ms/step - accuracy: 0.7657 - loss: 0.6086 - val_accuracy: 0.7603 - val_loss: 0.6194
Epoch 8/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 64s 180ms/step - accuracy: 0.7716 - loss: 0

In [113]:
test_loss13,test_accu13 = model13.evaluate(test_aug)
valid_loss13,valid_accu13=model13.evaluate(valid_aug)
training_loss13,training_accu13=model13.evaluate(train_aug)
print(f"validation Accuracy: {valid_accu13* 100:.2f}%")
print(f"validation Loss: {valid_loss13}")
print(f"training Accuracy: {training_accu13* 100:.2f}%")
print(f"training Loss: {training_loss13}")
print(f"Accuracy: {test_accu13* 100:.2f}%")
print("average training time is ",average_train_time_per_epoch13)

82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.8268 - loss: 0.7681
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.8101 - loss: 0.7637
357/357 ━━━━━━━━━━━━━━━━━━━━ 62s 173ms/step - accuracy: 0.8611 - loss: 0.4437
validation Accuracy: 80.61%
validation Loss: 0.7896363139152527
training Accuracy: 85.52%
training Loss: 0.45537200570106506
Accuracy: 79.73%
average training time is  64.92624915440878


In [117]:
model14=Sequential([
    Conv2D(32,(3,3),input_shape=(224,224,3),activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(64,(3,3),activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(128,(3,3),activation="relu"),
    
    
    Flatten(),
    Dense(64,activation="relu"),
    Dropout(0.3),
    Dense(4,activation="softmax"),

])
model14.summary()

Model: "sequential_25"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_55 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_39 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_56 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_40 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_57 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_25 (Flatten)            │ (None, 346112)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_51 (Dense)                │ (None, 64)             │    22,151,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_52 (Dense)                │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,244,740 (84.86 MB)

 Trainable params: 22,244,740 (84.86 MB)

 Non-trainable params: 0 (0.00 B)

In [118]:
start_time14=time.time()
model14.compile(optimizer=RMSprop(learning_rate=0.001),loss="categorical_crossentropy",metrics=['accuracy'])
model14.fit(train_aug,validation_data=valid_aug,epochs=15,batch_size=16)
average_train_time_per_epoch14=(time.time()-start_time14)/15

Epoch 1/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 69s 186ms/step - accuracy: 0.4572 - loss: 1.5973 - val_accuracy: 0.7420 - val_loss: 0.6767
Epoch 2/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 67s 187ms/step - accuracy: 0.7002 - loss: 0.7387 - val_accuracy: 0.7466 - val_loss: 0.6727
Epoch 3/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 181ms/step - accuracy: 0.7295 - loss: 0.6689 - val_accuracy: 0.7725 - val_loss: 0.5779
Epoch 4/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 181ms/step - accuracy: 0.7534 - loss: 0.6395 - val_accuracy: 0.7374 - val_loss: 0.7532
Epoch 5/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 183ms/step - accuracy: 0.7781 - loss: 0.5885 - val_accuracy: 0.7618 - val_loss: 0.6473
Epoch 6/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 66s 185ms/step - accuracy: 0.7877 - loss: 0.5432 - val_accuracy: 0.7374 - val_loss: 0.6151
Epoch 7/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 67s 188ms/step - accuracy: 0.8128 - loss: 0.4996 - val_accuracy: 0.8198 - val_loss: 0.4801
Epoch 8/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 181ms/step - accuracy: 0.8130 - loss: 0

In [122]:
test_loss14,test_accu14 = model14.evaluate(test_aug)
valid_loss14,valid_accu14=model14.evaluate(valid_aug)
training_loss14,training_accu14=model14.evaluate(train_aug)
print(f"validation Accuracy: {valid_accu14* 100:.2f}%")
print(f"validation Loss: {valid_loss14}")
print(f"training Accuracy: {training_accu14* 100:.2f}%")
print(f"training Loss: {training_loss14}")
print(f"Accuracy: {test_accu14* 100:.2f}%")
print("average training time is ",average_train_time_per_epoch14)

82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.8137 - loss: 0.5619
82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.8138 - loss: 0.5181
357/357 ━━━━━━━━━━━━━━━━━━━━ 62s 174ms/step - accuracy: 0.8481 - loss: 0.3664
validation Accuracy: 82.14%
validation Loss: 0.49135130643844604
training Accuracy: 84.93%
training Loss: 0.3748077154159546
Accuracy: 81.10%
average training time is  65.69570782979329


In [123]:
model15=Sequential([
    Conv2D(32,(3,3),input_shape=(224,224,3),activation="relu"),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Conv2D(64,(3,3),activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(128,(3,3),activation="relu"),
    
    
    Flatten(),
    Dense(64,activation="relu"),
    Dense(4,activation="softmax"),

])
model15.summary()

Model: "sequential_26"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_58 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 222, 222, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_41 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_59 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_42 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_60 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_26 (Flatten)            │ (None, 346112)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_53 (Dense)                │ (None, 64)             │    22,151,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_54 (Dense)                │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,244,868 (84.86 MB)

 Trainable params: 22,244,804 (84.86 MB)

 Non-trainable params: 64 (256.00 B)

In [124]:
start_time15=time.time()
model15.compile(optimizer=RMSprop(learning_rate=0.001),loss="categorical_crossentropy",metrics=['accuracy'])
model15.fit(train_aug,validation_data=valid_aug,epochs=15,batch_size=16)
average_train_time_per_epoch15=(time.time()-start_time15)/15

Epoch 1/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 69s 184ms/step - accuracy: 0.4758 - loss: 4.7650 - val_accuracy: 0.5863 - val_loss: 1.0326
Epoch 2/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 66s 184ms/step - accuracy: 0.7354 - loss: 0.6839 - val_accuracy: 0.8092 - val_loss: 0.5269
Epoch 3/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 182ms/step - accuracy: 0.7906 - loss: 0.5598 - val_accuracy: 0.7344 - val_loss: 0.8655
Epoch 4/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 181ms/step - accuracy: 0.8156 - loss: 0.4775 - val_accuracy: 0.8534 - val_loss: 0.4369
Epoch 5/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 64s 181ms/step - accuracy: 0.8336 - loss: 0.4432 - val_accuracy: 0.7023 - val_loss: 0.8697
Epoch 6/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 66s 184ms/step - accuracy: 0.8528 - loss: 0.4018 - val_accuracy: 0.8672 - val_loss: 0.3763
Epoch 7/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 64s 181ms/step - accuracy: 0.8646 - loss: 0.3749 - val_accuracy: 0.8611 - val_loss: 0.4548
Epoch 8/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 65s 181ms/step - accuracy: 0.8720 - loss: 0

In [125]:
test_loss15,test_accu15 = model15.evaluate(test_aug)
valid_loss15,valid_accu15=model15.evaluate(valid_aug)
training_loss15,training_accu15=model15.evaluate(train_aug)
print(f"validation Accuracy: {valid_accu15* 100:.2f}%")
print(f"validation Loss: {valid_loss15}")
print(f"training Accuracy: {training_accu15* 100:.2f}%")
print(f"training Loss: {training_loss15}")
print(f"Accuracy: {test_accu15* 100:.2f}%")
print("average training time is ",average_train_time_per_epoch15)

82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.8285 - loss: 0.5983
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.8252 - loss: 0.7910
357/357 ━━━━━━━━━━━━━━━━━━━━ 62s 174ms/step - accuracy: 0.8869 - loss: 0.3264
validation Accuracy: 83.05%
validation Loss: 0.6433514356613159
training Accuracy: 89.06%
training Loss: 0.31811976432800293
Accuracy: 82.01%
average training time is  65.05968521436056


In [16]:
model11=Sequential([
    Conv2D(32,(3,3),input_shape=(224,224,3),activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(64,(3,3),activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(128,(3,3),activation="relu"),
    
    
    Flatten(),
    Dense(64,activation="relu"),
    Dense(4,activation="softmax"),

])
model11.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 346112)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │    22,151,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,244,740 (84.86 MB)

 Trainable params: 22,244,740 (84.86 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
start_time11=time.time()
model11.compile(optimizer=RMSprop(learning_rate=0.001),loss="categorical_crossentropy",metrics=['accuracy'])
model11.fit(train_aug,validation_data=valid_aug,epochs=15,batch_size=16)
average_train_time_per_epoch11=(time.time()-start_time11)/15

Epoch 1/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 81s 221ms/step - accuracy: 0.4710 - loss: 1.5950 - val_accuracy: 0.4382 - val_loss: 2.7355
Epoch 2/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 79s 221ms/step - accuracy: 0.7082 - loss: 0.7442 - val_accuracy: 0.6840 - val_loss: 0.8080
Epoch 3/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 78s 218ms/step - accuracy: 0.7373 - loss: 0.6478 - val_accuracy: 0.7511 - val_loss: 0.6623
Epoch 4/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 79s 222ms/step - accuracy: 0.7780 - loss: 0.5497 - val_accuracy: 0.7908 - val_loss: 0.5813
Epoch 5/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 78s 219ms/step - accuracy: 0.7994 - loss: 0.5108 - val_accuracy: 0.7221 - val_loss: 0.9387
Epoch 6/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 77s 216ms/step - accuracy: 0.8205 - loss: 0.4771 - val_accuracy: 0.8107 - val_loss: 0.4405
Epoch 7/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 76s 213ms/step - accuracy: 0.8277 - loss: 0.4388 - val_accuracy: 0.8000 - val_loss: 0.4797
Epoch 8/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 78s 219ms/step - accuracy: 0.8355 - loss: 0

In [18]:
test_loss11,test_accu11 = model11.evaluate(test_aug)
valid_loss11,valid_accu11=model11.evaluate(valid_aug)
training_loss11,training_accu11=model11.evaluate(train_aug)
print(f"validation Accuracy: {valid_accu11* 100:.2f}%")
print(f"validation Loss: {valid_loss11}")
print(f"training Accuracy: {training_accu11* 100:.2f}%")
print(f"training Loss: {training_loss11}")
print(f"Accuracy: {test_accu11* 100:.2f}%")
print("average training time is ",average_train_time_per_epoch11)

82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.8404 - loss: 0.4104
82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.8987 - loss: 0.3121
357/357 ━━━━━━━━━━━━━━━━━━━━ 75s 209ms/step - accuracy: 0.8843 - loss: 0.2943
validation Accuracy: 88.55%
validation Loss: 0.3210659921169281
training Accuracy: 88.60%
training Loss: 0.30262014269828796
Accuracy: 86.28%
average training time is  78.08765853246054


In [20]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
predictions = model11.predict(test_aug, verbose=1)
y_pred = np.argmax(predictions, axis=1)#Tells argmax to pick the class with the max score per row (i.e., per sample)

82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step


In [21]:
print(predictions)


[[9.96004760e-01 5.96415077e-04 5.75852573e-05 3.34122521e-03]
 [9.99965429e-01 3.22563938e-05 3.55100269e-07 1.98509088e-06]
 [5.71904797e-03 2.03874009e-03 4.21639270e-04 9.91820574e-01]
 ...
 [6.23902399e-03 9.93654847e-01 3.06636025e-06 1.03082020e-04]
 [1.38346843e-12 1.00066885e-01 8.99933159e-01 1.43878594e-15]
 [9.83179212e-01 1.21860802e-02 1.75487308e-03 2.87985872e-03]]


In [22]:
print(y_pred)


[0 0 3 2 2 2 3 1 1 2 3 2 1 2 1 1 2 1 2 2 3 3 1 2 2 0 2 3 1 1 1 2 3 1 3 2 0
 3 2 3 2 2 2 1 1 0 3 3 0 2 1 2 3 3 1 2 1 2 2 3 2 2 3 3 3 1 3 3 1 1 3 1 2 3
 2 3 1 3 1 1 3 3 1 3 2 2 2 2 2 2 0 3 1 1 3 3 2 0 2 0 1 2 0 0 3 0 2 2 0 1 0
 1 1 3 1 0 0 3 0 0 1 2 1 3 2 1 2 2 2 2 2 0 0 1 3 3 2 2 1 0 2 1 2 3 2 1 1 2
 3 2 1 0 2 1 3 2 1 3 2 3 3 1 3 0 2 2 3 2 0 2 1 1 3 3 3 2 2 3 0 2 2 0 2 0 2
 3 3 3 2 3 0 1 1 1 1 2 3 1 2 2 3 2 3 2 1 3 1 1 2 2 0 3 3 1 1 1 3 2 2 1 3 3
 1 3 3 2 1 1 2 2 0 2 2 0 2 0 1 2 2 1 1 1 3 2 1 2 1 0 1 0 3 1 2 1 3 0 3 2 3
 1 3 1 1 3 2 2 3 3 3 2 0 0 1 2 2 2 0 0 1 2 1 2 1 2 3 2 1 1 0 2 1 1 3 3 2 3
 2 2 3 2 2 1 2 2 2 3 2 2 2 2 0 2 0 0 2 1 1 3 3 2 1 3 1 1 2 1 3 1 0 1 2 1 2
 3 3 2 0 2 3 2 2 3 1 0 2 1 1 2 1 2 2 0 0 2 1 2 1 1 2 2 2 1 3 3 0 0 3 2 3 2
 2 0 1 2 2 3 3 0 1 3 1 0 1 3 0 0 3 2 3 2 1 2 1 1 1 1 0 0 2 1 3 2 1 2 1 2 2
 1 1 2 1 0 2 0 2 3 3 3 3 2 3 2 0 1 2 0 1 0 3 3 1 0 1 2 1 2 2 2 2 2 0 1 1 1
 3 3 2 0 2 0 2 2 1 0 2 2 3 2 2 1 2 3 0 3 1 3 3 0 1 3 2 1 0 2 2 3 1 2 1 0 0
 0 2 2 0 3 1 2 3 0 2 3 3 

In [23]:
y_true = test_aug.classes

In [24]:
y_true


[0,
 0,
 3,
 2,
 2,
 1,
 3,
 0,
 1,
 2,
 3,
 2,
 0,
 2,
 1,
 1,
 2,
 0,
 2,
 2,
 3,
 3,
 1,
 1,
 2,
 0,
 2,
 3,
 1,
 1,
 0,
 2,
 3,
 1,
 3,
 2,
 0,
 3,
 2,
 0,
 1,
 2,
 2,
 1,
 1,
 0,
 3,
 0,
 0,
 2,
 2,
 2,
 3,
 3,
 1,
 2,
 1,
 2,
 2,
 3,
 1,
 1,
 3,
 3,
 1,
 0,
 3,
 3,
 1,
 1,
 3,
 0,
 2,
 3,
 2,
 3,
 1,
 3,
 1,
 1,
 3,
 3,
 0,
 3,
 2,
 2,
 2,
 1,
 2,
 2,
 0,
 3,
 0,
 1,
 1,
 3,
 2,
 0,
 2,
 0,
 0,
 2,
 0,
 0,
 1,
 0,
 2,
 2,
 0,
 1,
 0,
 0,
 1,
 3,
 0,
 0,
 0,
 3,
 0,
 0,
 1,
 2,
 0,
 3,
 2,
 1,
 2,
 2,
 2,
 2,
 2,
 0,
 0,
 1,
 3,
 3,
 2,
 2,
 0,
 0,
 2,
 0,
 2,
 3,
 2,
 1,
 1,
 2,
 3,
 2,
 1,
 0,
 1,
 0,
 3,
 2,
 1,
 3,
 2,
 3,
 3,
 1,
 3,
 0,
 2,
 2,
 3,
 2,
 0,
 2,
 0,
 1,
 3,
 3,
 3,
 1,
 2,
 3,
 0,
 2,
 2,
 0,
 2,
 0,
 2,
 3,
 0,
 3,
 1,
 3,
 0,
 0,
 1,
 0,
 0,
 2,
 3,
 1,
 1,
 2,
 3,
 2,
 3,
 2,
 1,
 3,
 1,
 0,
 2,
 2,
 0,
 3,
 1,
 1,
 0,
 1,
 3,
 2,
 2,
 1,
 3,
 1,
 1,
 3,
 3,
 2,
 0,
 1,
 2,
 2,
 0,
 2,
 2,
 0,
 2,
 0,
 1,
 2,
 2,
 1,
 0,
 1,
 3,
 2,
 0,
 2,
 1,
 0,
 1,
 0,


In [25]:
test_aug.class_indices

{'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}

In [26]:
test_aug.classes

[0,
 0,
 3,
 2,
 2,
 1,
 3,
 0,
 1,
 2,
 3,
 2,
 0,
 2,
 1,
 1,
 2,
 0,
 2,
 2,
 3,
 3,
 1,
 1,
 2,
 0,
 2,
 3,
 1,
 1,
 0,
 2,
 3,
 1,
 3,
 2,
 0,
 3,
 2,
 0,
 1,
 2,
 2,
 1,
 1,
 0,
 3,
 0,
 0,
 2,
 2,
 2,
 3,
 3,
 1,
 2,
 1,
 2,
 2,
 3,
 1,
 1,
 3,
 3,
 1,
 0,
 3,
 3,
 1,
 1,
 3,
 0,
 2,
 3,
 2,
 3,
 1,
 3,
 1,
 1,
 3,
 3,
 0,
 3,
 2,
 2,
 2,
 1,
 2,
 2,
 0,
 3,
 0,
 1,
 1,
 3,
 2,
 0,
 2,
 0,
 0,
 2,
 0,
 0,
 1,
 0,
 2,
 2,
 0,
 1,
 0,
 0,
 1,
 3,
 0,
 0,
 0,
 3,
 0,
 0,
 1,
 2,
 0,
 3,
 2,
 1,
 2,
 2,
 2,
 2,
 2,
 0,
 0,
 1,
 3,
 3,
 2,
 2,
 0,
 0,
 2,
 0,
 2,
 3,
 2,
 1,
 1,
 2,
 3,
 2,
 1,
 0,
 1,
 0,
 3,
 2,
 1,
 3,
 2,
 3,
 3,
 1,
 3,
 0,
 2,
 2,
 3,
 2,
 0,
 2,
 0,
 1,
 3,
 3,
 3,
 1,
 2,
 3,
 0,
 2,
 2,
 0,
 2,
 0,
 2,
 3,
 0,
 3,
 1,
 3,
 0,
 0,
 1,
 0,
 0,
 2,
 3,
 1,
 1,
 2,
 3,
 2,
 3,
 2,
 1,
 3,
 1,
 0,
 2,
 2,
 0,
 3,
 1,
 1,
 0,
 1,
 3,
 2,
 2,
 1,
 3,
 1,
 1,
 3,
 3,
 2,
 0,
 1,
 2,
 2,
 0,
 2,
 2,
 0,
 2,
 0,
 1,
 2,
 2,
 1,
 0,
 1,
 3,
 2,
 0,
 2,
 1,
 0,
 1,
 0,


In [29]:
class_labels = list(test_aug.class_indices.keys())
class_labels

['glioma', 'meningioma', 'notumor', 'pituitary']

In [30]:
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[ 96  49   0   4]
 [  0 117  21  10]
 [  0   4 204   0]
 [  1   1   0 149]]


Rows: actual classes

Columns: predicted classes

Diagonal: correct predictions

Off-diagonal: misclassifications

In [31]:
cr = classification_report(y_true, y_pred, target_names=class_labels)
print("Classification Report:")
print(cr)

Classification Report:
              precision    recall  f1-score   support

      glioma       0.99      0.64      0.78       149
  meningioma       0.68      0.79      0.73       148
     notumor       0.91      0.98      0.94       208
   pituitary       0.91      0.99      0.95       151

    accuracy                           0.86       656
   macro avg       0.87      0.85      0.85       656
weighted avg       0.88      0.86      0.86       656

